In [1]:
from dotenv import load_dotenv
load_dotenv()

False

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = ""

In [3]:
from langchain_core.tools import tool

In [9]:
@tool
def get_credit_score(pan: str) -> int:
    """Fetch the CIBIL credit score for a customer using their PAN number.
    Use this whenever a credit decision needs a bureau score.
    input param : PAN
    output value : Number (cibil score)
    """
    fake_bureau = {"ABCDE1234F": 762, "XYZAB9876K": 640}
    return fake_bureau.get(pan, 700)

In [10]:
@tool
def calculate_foir(monthly_income: float, existing_emi: float) -> float:
    """Calculate FOIR (Fixed Obligation to Income Ratio) as a percentage.
    Use this to check whether the applicant's existing EMIs are within bank policy."""
    return round((existing_emi / monthly_income) * 100, 1)


In [12]:
for t in (get_credit_score, calculate_foir):
    print(f"name   : {t.name}")
    print(f"desc   : {t.description}")
    print(f"schema : {t.args}\n")

name   : get_credit_score
desc   : Fetch the CIBIL credit score for a customer using their PAN number.
Use this whenever a credit decision needs a bureau score.
input param : PAN
output value : Number (cibil score)
schema : {'pan': {'title': 'Pan', 'type': 'string'}}

name   : calculate_foir
desc   : Calculate FOIR (Fixed Obligation to Income Ratio) as a percentage.
Use this to check whether the applicant's existing EMIs are within bank policy.
schema : {'monthly_income': {'title': 'Monthly Income', 'type': 'number'}, 'existing_emi': {'title': 'Existing Emi', 'type': 'number'}}



In [17]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from langchain_openai import ChatOpenAI

In [18]:
chat = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.3)

In [15]:
def call_model(state: MessagesState) -> dict:
    # state["messages"] already holds the FULL history for this thread
    return {"messages": [chat.invoke(state["messages"])]}

In [25]:
builder = StateGraph(MessagesState)
builder.add_node("model", call_model)

builder.add_edge(START, "model")
builder.add_edge("model", END)

In [26]:
app = builder.compile(checkpointer=InMemorySaver())

In [27]:
config = {"configurable": {"thread_id": "customer-101"}}   # <-- the memory key

In [28]:
result = app.invoke({"messages": [{"role":"user", "content": "My name is Mani"}]}, config)
for m in result["messages"]:
    print(f"{m.type} | {m.content}")

human | My name is Mani
ai | Nice to meet you, Mani! How can I assist you today?


In [29]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

In [30]:
agent = create_agent(
    model="openai:gpt-3.5-turbo",          # or "bedrock_converse:<model-id>"
    tools=[get_credit_score, calculate_foir],
    system_prompt=(
        "You are a loan eligibility assistant for an Indian retail bank. "
        "Policy: minimum credit score 700, maximum FOIR 50%. "
        "Always fetch the score and compute FOIR before deciding."
    ),
    checkpointer=InMemorySaver()
)

In [31]:
config = {"configurable": {"thread_id": "app-9001"}}

In [32]:
out = agent.invoke(
    {"messages": [{"role": "user", "content":
        "PAN ABCDE1234F, monthly income 90000, existing EMI 30000. Eligible?"}]},
    config,
)

In [33]:
print(out["messages"][-1].text)

The credit score fetched for PAN ABCDE1234F is 762, and the FOIR (Fixed Obligation to Income Ratio) is 33.3%. 

Since the credit score is above the minimum requirement of 700 and the FOIR is within the bank's policy of 50%, the applicant is eligible.


In [34]:
trace_config = {"configurable": {"thread_id": "app-9002"}}  # fresh thread so we start clean

In [35]:
seen = 0
for step in agent.stream(
    {"messages": [{"role": "user", "content":
        "PAN ABCDE1234F, monthly income 90000, existing EMI 30000. Eligible?"}]},
    trace_config,
    stream_mode="values",
):
    messages = step["messages"]
    for m in messages[seen:]:
        print(f"--- {m.type} ---")
        if getattr(m, "tool_calls", None):
            for tc in m.tool_calls:
                print(f"  tool call -> {tc['name']}({tc['args']})")
        if m.content:
            print(f"  {m.content}")
    seen = len(messages)

--- human ---
  PAN ABCDE1234F, monthly income 90000, existing EMI 30000. Eligible?
--- ai ---
  tool call -> get_credit_score({'pan': 'ABCDE1234F'})
  tool call -> calculate_foir({'monthly_income': 90000, 'existing_emi': 30000})
--- tool ---
  762
--- tool ---
  33.3
--- ai ---
  The credit score fetched is 762 and the FOIR is 33.3%. 

Since the credit score is above the minimum requirement of 700 and the FOIR is within the policy limit of 50%, the applicant is eligible for the loan.
